In [1]:
import pandas as pd
import tensorflow as tf 
import os
import resampy
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import datetime
import argparse
import h5py

# Model specific
import params as yamnet_params
import yamnet as yamnet_model

In [7]:
input_folder = ""
output_filename = ".h5"
dataset_file = "../audioset_eval_strong_filt_downloaded.csv"

#### Load Base model 

# The graph is designed for a sampling rate of 16 kHz, but higher rates should work too.
# We also generate scores at a 10 Hz frame rate.
sr = 16000
params = yamnet_params.Params(sample_rate=sr, patch_hop_seconds=1)
print("Sample rate =", params.sample_rate)

# Set up the YAMNet model.
class_names = yamnet_model.class_names('yamnet_class_map.csv')
yamnet = yamnet_model.yamnet_frames_model(params)
yamnet.load_weights('yamnet.h5')


Sample rate = 16000


In [30]:
df_class_map = pd.read_csv("yamnet_class_map.csv")
df_class_map

mid_dict = {mid:i for i, mid in enumerate(df_class_map["mid"].values)}
index_dict = {i:mid for i, mid in enumerate(df_class_map["mid"].values)}

def mid_to_index(mid):
    return mid_dict[mid]

def index_to_mid(mid):
    return index_dict[mid]



In [32]:
print("/m/02qldy")

/m/02qldy


In [23]:
df_class_map

,index,mid,display_name
0,0,/m/09x0r,Speech
1,1,/m/0ytgt,"Child speech, kid speaking"
2,2,/m/01h8n0,Conversation
3,3,/m/02qldy,"Narration, monologue"
4,4,/m/0261r1,Babbling
...,...,...,...
516,516,/m/07p_0gm,Throbbing
517,517,/m/01jwx6,Vibration
518,518,/m/07c52,Television
519,519,/m/06bz3,Radio


In [8]:
def load_audio(filename):
    x, _ = sf.read(filename)
    if len(x.shape) > 1:
        x = np.mean(x, axis=1)
    return x

def predict(x):
    return ""

In [33]:
##### Read Database

df_metadata = pd.read_csv(dataset_file)
df_metadata["label_yamnet"] = df_metadata.apply(lambda x: mid_to_index(x["label"]),  axis=1)


In [36]:
all_scores = []
ground_truth = []

for i,clip in tqdm(df_metadata.iterrows()):
    x = load_audio(clip["path"])
    scores, embeddings, spectrogram = yamnet(x)
    scores = scores.numpy()
    mean_scores = np.mean(scores, axis=0)
    all_scores.append(mean_scores)
    ground_truth.append(clip["label_yamnet"])
    

3368it [06:44,  5.77it/s]

: 

: 

Metrics

In [60]:
predictions = h5py.File("yamnet_predictions_eval_audioset.h5","r")
print(predictions.keys())
n_classes = 521
y = predictions["y"][...]
y_test = np.eye(len(y))[y]
y_hat = predictions["y_hat"][...]



<KeysViewHDF5 ['y', 'y_hat']>


In [61]:

predictions = np.argmax(y_hat,axis=1)
y_hat_one = np.eye(len(y_hat))[predictions]


In [62]:
from sklearn.metrics import average_precision_score
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_hat_one)

0.04911465427449724

In [63]:
from sklearn.metrics import PrecisionRecallDisplay


display = PrecisionRecallDisplay.from_predictions(y_test, y_hat_one, name="YAMNET baseline")
_ = display.ax_.set_title("2-class Precision-Recall curve")

ValueError: multilabel-indicator format is not supported

In [18]:
y_hat.shape

(24962, 521)